# First Colonies Detection - ΔrhlAΔpqsL Replicate 2

This notebook is used to manually identify and record the coordinates of first regrowth colonies from microscopy time-lapse data.

**Workflow:**
1. Open time-lapse TIFF files in Napari for visual inspection
2. Identify chambers with regrowth events
3. Crop regions of interest for detailed analysis
4. Manually mark coordinates of first colonies using Napari points layer
5. Convert coordinates to micrometers and assign spatial bins
6. Store data for all positions and save to CSV

**Output:**
- `0_first_colonies_rhlApqsL_rep2.csv`: Coordinates and metadata of first colonies for replicate 2

**Note:** This is an interactive notebook requiring manual annotation in Napari

**⚠️ Data Requirement — BioImageArchive:** This notebook requires raw microscopy data from BioImageArchive. Download the dataset and set `data_archive_path` to your local BioImageArchive directory (see README).

**Pipeline step 1/2 (replicate 2)** — run before `1_plot_first_colonies_rhlApqsL_all_rep.ipynb`.

**Run analysis scripts in this order:**
1. `0_first_colonies_rhlApqsL_rep1/2/3.ipynb` — load BioImageArchive data and annotate first colonies (run for each replicate)
2. `1_plot_first_colonies_rhlApqsL_all_rep.ipynb` — merge replicates and calculate regrowth fractions

## Import Libraries

In [ ]:
import napari
import tifffile
import dask.array as da
import zarr
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
#set path to BioImageArchive data directory:
data_archive_path = '/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/'
base_path = os.path.join(data_archive_path, 'ToleranceAssay/2_DrhlA-DpqsL/replicate_2')

## Step 1: Browse TIFF Files in Napari

Open all TIFF files from replicate 2 in Napari to visually identify chambers with regrowth events

In [ ]:
# Define relative path to Image_Data directory
tif_files = [os.path.join(base_path, f"20250415_pqsLrhlA_pos{i}_mcherry.tif") for i in range(0, 9)]

for tif_path in tif_files:
    print(f"Opening {tif_path}...")
    tif = tifffile.TiffFile(tif_path)  # keep open
    zarr_store = tif.aszarr()
    image_lazy = da.from_zarr(zarr.open(zarr_store, mode='r'))

    viewer = napari.Viewer()
    viewer.add_image(image_lazy, name=os.path.basename(tif_path), colormap='magenta')
    napari.run()

    tif.close()  # close after viewing


## Step 2: Preview Chamber Crop

Preview crop region to confirm it captures the chamber with regrowth event

In [ ]:
# Your chosen file and frame (update for each position)
tif_path = os.path.join(base_path, "20250415_pqsLrhlA_pos9_mcherry.tif")
frame_index = 180  # change to your frame

# Crop coordinates
y_start, y_end = 930, 1630
x_start, x_end = 640, 1390

# Load only that frame
with tifffile.TiffFile(tif_path) as tif:
    frame = tif.asarray(key=frame_index)

# Show full frame with crop rectangle
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(frame, cmap='gray')
rect = plt.Rectangle((x_start, y_start),
                     x_end - x_start,
                     y_end - y_start,
                     edgecolor='red', facecolor='none', linewidth=2)
ax.add_patch(rect)
ax.set_title(f"Frame {frame_index} with crop preview")
plt.show()

# Show cropped region
crop = frame[y_start:y_end, x_start:x_end]
plt.figure(figsize=(6, 6))
plt.imshow(crop, cmap='gray')
plt.title("Cropped region")
plt.show()


## Step 3: Initialize Data Storage

Initialize empty list to store colony coordinates from all positions. **Run this cell only once at the start!**

In [ ]:
# Initialize empty list - RUN ONLY ONCE!
stored_data = []

print("✓ Storage initialized. Ready to add positions.")

## Step 4: Annotate Colonies in Napari

Open cropped region in Napari and manually mark first colony positions using the points layer

In [ ]:
# ==========================================
# CHANGE THESE for each position
# ==========================================
tif_path = os.path.join(base_path, "20250415_pqsLrhlA_pos0_mcherry.tif")
frame_index = 5
replicate = "replicate2"

# Crop coordinates
y_start, y_end = 990, 1700
x_start, x_end = 700, 1450

# Load cropped frame
with tifffile.TiffFile(tif_path) as tif:
    frame = tif.asarray(key=frame_index)
crop = frame[y_start:y_end, x_start:x_end]

# Open in Napari
viewer = napari.Viewer()
viewer.add_image(crop, name='Cropped ROI', colormap='magenta')
regrowth_layer = viewer.add_points(
    np.empty((0, 2)),
    name='Regrowth Points',
    face_color='red',
    size=8
)

napari.run()

## Step 5: Process and Store Coordinates

Convert marked coordinates to micrometers, assign spatial bins, and append to storage list

In [ ]:
# --- After closing Napari ---
coords_relative = regrowth_layer.data

# Settings
pixel_to_um = 0.065
minutes_per_frame = 5
num_bins = 10
bin_edges = np.linspace(0, 50, num_bins + 1)
position_name = os.path.basename(tif_path).split("_")[1]

# Build DataFrame
if len(coords_relative) == 0:
    df = pd.DataFrame({
        'colony_id': [0],
        'x': [np.nan],
        'y': [np.nan],
        'frame': [frame_index],
        'position': [position_name],
        'replicate': [replicate],
        'hours': [frame_index * (minutes_per_frame / 60)],
        'x_um': [np.nan],
        'y_um': [np.nan],
        'y_bin': [np.nan],
        'x_bin': [np.nan],
        'regrowth': ['no']
    })
    print(f"✓ {position_name} frame {frame_index}: No regrowth")
else:
    df = pd.DataFrame({
        'y': coords_relative[:, 0],
        'x': coords_relative[:, 1],
    })
    
    df['colony_id'] = np.arange(1, len(df) + 1)
    df['frame'] = frame_index
    df['position'] = position_name
    df['replicate'] = replicate
    df['hours'] = df['frame'] * (minutes_per_frame / 60)
    df['x_um'] = df['x'] * pixel_to_um
    df['y_um'] = df['y'] * pixel_to_um
    df['y_bin'] = pd.cut(df['y_um'], bins=bin_edges)
    df['x_bin'] = pd.cut(df['x_um'], bins=bin_edges)
    df['regrowth'] = 'yes'
    
    df = df[['colony_id', 'x', 'y', 'frame', 'position', 'replicate',
             'hours', 'x_um', 'y_um', 'y_bin', 'x_bin', 'regrowth']]
    
    print(f"✓ {position_name} frame {frame_index}: Added {len(df)} points")

# APPEND to stored_data
stored_data.append(df)

print(f"📦 Total positions stored: {len(stored_data)}")

## Step 6: Save to CSV

Combine all stored data and save to CSV file

In [ ]:
# ==========================================
# Save all stored data to CSV
# ==========================================

# Combine all DataFrames
combined_df = pd.concat(stored_data, ignore_index=True)

print(f"\n📊 Summary:")
print(f"  Total entries: {len(combined_df)}")
print(f"  Positions analyzed: {combined_df['position'].nunique()}")
print(f"  Regrowth detected: {(combined_df['regrowth'] == 'yes').sum()} points")
print(f"  No regrowth: {(combined_df['regrowth'] == 'no').sum()} positions")

# Display preview
print("\nPreview of data:")
print(combined_df)

# Save to CSV using relative path
csv_path = "0_first_colonies_rhlApqsL_rep2.csv"

file_exists = os.path.exists(csv_path)

combined_df.to_csv(
    csv_path,
    mode='a',
    header=not file_exists,
    index=False
)

print(f"\n✅ Saved {len(combined_df)} rows to {csv_path}")